# PyBaMM 电池仿真入门示例

本 Notebook 包含 **6 个循序渐进的仿真示例**，帮助你快速上手 PyBaMM 电池建模仿真。

**运行前确认**: 已激活 `pybamm` conda 环境。

In [ ]:
import pybamm

print(f"PyBaMM 版本: {pybamm.__version__}")
print(f"PyBaMM 安装路径: {pybamm.__path__[0]}")

---
## 示例 1: 最简单的仿真 — SPM 模型恒流放电

这是 PyBaMM 最基本的使用方式，4 行代码完成一个仿真：
- **SPM (Single Particle Model)**: 最快的电池模型，假设正负电极颗粒内浓度均匀
- 仿真时间: 3600 秒 = 1 小时
- 默认 1C 倍率放电

In [ ]:
model = pybamm.lithium_ion.SPM()
sim = pybamm.Simulation(model)
sim.solve([0, 3600])
sim.plot()

---
## 示例 2: 对比三种模型

将 **SPM**、**SPMe** 和 **DFN** 三种模型的放电曲线放在一起对比：
- SPM: 单粒子模型（最快，精度最低）
- SPMe: 单粒子 + 电解液模型（平衡精度与速度）
- DFN: Doyle-Fuller-Newman 模型（最精确，最慢）

In [ ]:
models = [
    pybamm.lithium_ion.SPM(),
    pybamm.lithium_ion.SPMe(),
    pybamm.lithium_ion.DFN(),
]
sims = []
for m in models:
    sim = pybamm.Simulation(m)
    sim.solve([0, 3600])
    sims.append(sim)

pybamm.dynamic_plot(sims, labels=["SPM", "SPMe", "DFN"])

---
## 示例 3: 使用不同参数集

不同参数集对应不同的电池材料和设计。这里对比两个常用参数集的放电曲线：
- **Chen2020**: 石墨 / LiFePO4 电池
- **Marquis2019**: 石墨 / NMC532 电池

In [ ]:
model = pybamm.lithium_ion.SPMe()

sims = []
for param_name in ["Chen2020", "Marquis2019"]:
    param = pybamm.ParameterValues(param_name)
    sim = pybamm.Simulation(model, parameter_values=param)
    sim.solve([0, 3600])
    sims.append(sim)

pybamm.dynamic_plot(sims, labels=["Chen2020 (Graphite/LFP)", "Marquis2019 (Graphite/NMC532)"])

---
## 示例 4: CCCV 充放电循环实验

模拟真实的 CCCV 充放电协议：
1. C/10 放电至 3.3 V
2. 静置 1 小时
3. 1A 恒流充电至 4.1 V
4. 4.1 V 恒压充电至电流降至 50 mA
5. 静置 1 小时
6. 重复 3 个循环

In [ ]:
cycle = [
    "Discharge at C/10 for 10 hours or until 3.3 V",
    "Rest for 1 hour",
    "Charge at 1 A until 4.1 V",
    "Hold at 4.1 V until 50 mA",
    "Rest for 1 hour",
]

experiment = pybamm.Experiment(cycle * 3)  # type: ignore[arg-type]

model = pybamm.lithium_ion.DFN()
sim = pybamm.Simulation(model, experiment=experiment)
sim.solve()
sim.plot()

---
## 示例 5: 不同倍率放电对比

对比 0.5C、1C、2C、3C 四种倍率下的放电曲线，观察高倍率对电压平台和容量的影响。

In [ ]:
model = pybamm.lithium_ion.SPMe()

C_rates = [0.5, 1, 2, 3]
sims = []

for C in C_rates:
    experiment = pybamm.Experiment(
        [f"Discharge at {C}C for 10 hours or until 2.5 V"]  # type: ignore[arg-type]
    )
    sim = pybamm.Simulation(model, experiment=experiment)
    sim.solve()
    sims.append(sim)

pybamm.dynamic_plot(sims, labels=[f"{C}C" for C in C_rates])

---
## 示例 6: 提取与导出仿真数据

从仿真结果中提取数值数据，并导出为 CSV 文件供后续分析使用。

In [ ]:
model = pybamm.lithium_ion.SPM()
sim = pybamm.Simulation(model)
solution = sim.solve([0, 3600])  # type: ignore[assignment]

import numpy as np
import pandas as pd

time = solution["Time [s]"].entries.flatten()
voltage = solution["Terminal voltage [V]"].entries.flatten()
current = solution["Current [A]"].entries.flatten()

# 辅助函数：将任意形状的数组降为一维，与 time 对齐
def to_1d(arr, target_len):
    arr = np.asarray(arr)
    if arr.ndim == 0:
        return np.full(target_len, arr.item())
    if arr.ndim == 1 and len(arr) == target_len:
        return arr
    # 多维数组：沿非时间维度取均值，再确保长度匹配
    if arr.ndim > 1:
        arr = arr.mean(axis=tuple(range(1, arr.ndim)))
    # 如果长度仍不匹配，用插值对齐
    if len(arr) != target_len:
        arr = np.interp(np.linspace(0, 1, target_len), np.linspace(0, 1, len(arr)), arr)
    return arr

neg_conc = to_1d(solution["Negative particle surface concentration [mol.m-3]"].entries, len(time))
pos_conc = to_1d(solution["Positive particle surface concentration [mol.m-3]"].entries, len(time))

print(f"仿真时间范围: {time[0]:.1f} ~ {time[-1]:.1f} 秒")
print(f"电压范围: {voltage.min():.3f} ~ {voltage.max():.3f} V")
print(f"数据点数: {len(time)}")
print(f"neg_conc shape: {neg_conc.shape}, pos_conc shape: {pos_conc.shape}")

df = pd.DataFrame({
    "Time [s]": time,
    "Terminal voltage [V]": voltage,
    "Current [A]": current,
    "Negative particle surface concentration [mol.m-3]": neg_conc,
    "Positive particle surface concentration [mol.m-3]": pos_conc,
})
df.to_csv("simulation_result.csv", index=False)
print("\n数据已保存到 simulation_result.csv")

In [ ]:
df.head(10)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(df["Time [s]"], df["Terminal voltage [V]"])
axes[0].set_xlabel("Time [s]")
axes[0].set_ylabel("Terminal Voltage [V]")
axes[0].set_title("Discharge Curve")
axes[0].grid(True, alpha=0.3)

axes[1].plot(df["Time [s]"], df["Current [A]"])
axes[1].set_xlabel("Time [s]")
axes[1].set_ylabel("Current [A]")
axes[1].set_title("Discharge Current")
axes[1].grid(True, alpha=0.3)

axes[2].plot(df["Time [s]"], df["Negative particle surface concentration [mol.m-3]"], label="Negative")
axes[2].plot(df["Time [s]"], df["Positive particle surface concentration [mol.m-3]"], label="Positive")
axes[2].set_xlabel("Time [s]")
axes[2].set_ylabel("Surface Concentration [mol/m3]")
axes[2].set_title("Particle Surface Concentration")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 完成！

你已经学会了 PyBaMM 仿真的基本操作。接下来可以：

1. 查看 `examples/scripts/` 目录下约 50 个进阶示例
2. 查看 `docs/source/examples/notebooks/` 目录下的 Jupyter 示例
3. 阅读 `SIMULATION_GUIDE.md` 了解更多细节
4. 访问在线文档: https://docs.pybamm.org